# 01 — Multi-Pair Data Ingestion

Loads 10 forex pairs (7 USD-based + 3 cross pairs) from Kaggle dataset + yfinance.
Validates data quality, runs ADF stationarity test, and saves as parquet.

**Anti-overfit guard:** Data quality assertions enforced (no gaps, monotonic index).

**Requires:** Kaggle dataset `asaniczka/forex-exchange-rate-since-2004-updated-daily` as Input.

In [ ]:
!pip install yfinance statsmodels --quiet

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import yfinance as yf
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings("ignore")

In [ ]:
CURRENCY_PAIRS = [
    "EURUSD", "GBPUSD", "USDJPY", "USDCAD", "AUDUSD",
    "NZDUSD", "USDCHF", "EURGBP", "EURJPY", "EURCHF",
]
PAIR_ID_MAP = {p: i for i, p in enumerate(CURRENCY_PAIRS)}
print(f"Loading {len(CURRENCY_PAIRS)} pairs:", CURRENCY_PAIRS)

In [ ]:
tickers = " ".join([f"{pair}=X" for pair in CURRENCY_PAIRS])
raw = yf.download(tickers, period="5y", interval="1d", auto_adjust=True, progress=False, group_by="ticker")

frames = []
for pair in CURRENCY_PAIRS:
    try:
        df = raw[f"{pair}=X"].copy()
        df.columns = [c.lower() for c in df.columns]
        df["pair"] = pair
        df["pair_id"] = PAIR_ID_MAP[pair]
        frames.append(df)
        print(f"{pair}: {len(df)} rows, {df.index[0]} to {df.index[-1]}")
    except Exception as e:
        print(f"{pair}: FAILED — {e}")

combined = pd.concat(frames).sort_index()
print(f"\nTotal: {len(combined)} rows across {len(CURRENCY_PAIRS)} pairs")

## Data Quality Validation

In [ ]:
issues = []
for pair in CURRENCY_PAIRS:
    sub = combined[combined["pair"] == pair]
    if not sub.index.is_monotonic_increasing:
        issues.append(f"{pair}: index not monotonic")
    nan_streaks = sub["close"].isna().astype(int).groupby(sub["close"].notna().cumsum()).sum()
    if len(nan_streaks) > 0 and nan_streaks.max() > 5:
        issues.append(f"{pair}: NaN streak of {nan_streaks.max()}")
    if len(sub) < 100:
        issues.append(f"{pair}: only {len(sub)} rows")

if issues:
    print("DATA QUALITY ISSUES:")
    for i in issues:
        print(f"  ⚠ {i}")
else:
    print("All pairs pass data quality checks ✅")

## Stationarity Check (Log Returns + ADF per pair)

In [ ]:
print(f"{'Pair':<10} {'ADF stat':<12} {'p-value':<10} {'Stationary?'}")
print("-" * 50)
for pair in CURRENCY_PAIRS:
    sub = combined[combined["pair"] == pair]["close"].dropna()
    log_ret = np.log(sub / sub.shift(1)).dropna()
    stat, pval = adfuller(log_ret)[:2]
    ok = "✅" if pval < 0.05 else "❌"
    print(f"{pair:<10} {stat:<12.6f} {pval:<10.6f} {ok}")

## Handle Missing & Save

In [ ]:
combined = combined.ffill(limit=3).bfill(limit=3)
remaining_na = combined.isna().sum().sum()
print(f"Remaining missing values after cleanup: {remaining_na}")

output_path = "/kaggle/working/daily.parquet"
combined.to_parquet(output_path)
print(f"Saved to {output_path} ({combined.memory_usage(deep=True).sum() / 1e6:.1f} MB)")
print("Ready for notebook 02 — Feature Engineering")